# Sales CSV - Silver transformation

**Medallion role:** Silver. This notebook standardizes inventory transactions, enriches them with product attributes, applies business-quality rules, and separates valid movements from rejected records. The resulting contract supports inventory analytics without losing diagnostic evidence.

In [0]:
dbutils.widgets.removeAll()

## Environment-specific Bronze inputs and Silver outputs

The selected catalog supplies two Bronze inputs: product reference data and inventory transactions. Accepted and rejected results are stored as external Delta tables at distinct paths so quality remediation remains independent from analytical consumption.

In [0]:
from datetime import datetime, timezone

dbutils.widgets.text(
    "environment",
    "dev",
    "Environment"
)

dbutils.widgets.text(
    "ingestion_timestamp",
    datetime.now(timezone.utc).isoformat(),
    "Ingestion Timestamp"
)

environment = dbutils.widgets.get("environment").lower()

ingestion_timestamp = (
    dbutils.widgets.get("ingestion_timestamp").strip()
)

if environment not in ["dev", "prod"]:
    raise ValueError(
        "Environment must be either 'dev' or 'prod'."
    )

config = {
    "dev": {
        "storage_account": "stcentralusjrdev",
        "catalog": "salescsv_dev"
    },
    "prod": {
        "storage_account": "stcentralusjrprod",
        "catalog": "salescsv_prod"
    }
}

env = config[environment]

storage_account = env["storage_account"]
catalog = env["catalog"]

product_bronze_table = (
    f"{catalog}.bronze.product_catalog_raw"
)

inventory_bronze_table = (
    f"{catalog}.bronze.inventory_transactions_raw"
)

silver_table = (
    f"{catalog}.silver.inventory_movements"
)

rejected_table = (
    f"{catalog}.silver.rejected_transactions"
)

silver_path = (
    f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
    "salescsv/silver/inventory_movements/"
)

rejected_path = (
    f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
    "salescsv/silver/rejected_transactions/"
)

print("=" * 60)
print("SALES CSV - SILVER TRANSFORMATION")
print("=" * 60)
print(f"Environment           : {environment}")
print(f"Product source        : {product_bronze_table}")
print(f"Inventory source      : {inventory_bronze_table}")
print(f"Silver target         : {silver_table}")
print(f"Rejected target       : {rejected_table}")
print("=" * 60)

In [0]:


product_bronze_df = spark.table(
    product_bronze_table
)

inventory_bronze_df = spark.table(
    inventory_bronze_table
)


## Reference cleanup, safe parsing, and enrichment

Product fields are normalized into a reusable lookup, while transaction values are standardized and parsed with tolerant functions that convert malformed input to `NULL` for explicit validation. A LEFT JOIN keeps every transaction in scope even when its product is missing, allowing the quality rules to flag that condition instead of dropping the record.

In [0]:


from pyspark.sql.functions import (
    col,
    trim,
    upper,
    initcap
)


product_lookup_df = (
    product_bronze_df

    .select(
        upper(
            trim(col("product_id"))
        ).alias("product_id"),

        trim(
            col("product_name")
        ).alias("product_name"),

        initcap(
            trim(col("category"))
        ).alias("category"),

        trim(
            col("supplier")
        ).alias("supplier"),

        col("unit_cost")
            .cast("decimal(12,2)")
            .alias("unit_cost"),

        col("reorder_level")
            .cast("int")
            .alias("reorder_level")
    )

    .dropDuplicates(
        ["product_id"]
    )
)

In [0]:
from pyspark.sql.functions import (
    expr,
    lit
)


inventory_prepared_df = (
    inventory_bronze_df

    .withColumn(
        "transaction_id",
        upper(trim(col("transaction_id")))
    )

    .withColumn(
        "product_id",
        upper(trim(col("product_id")))
    )

    .withColumn(
        "warehouse",
        initcap(trim(col("warehouse")))
    )

    .withColumn(
        "transaction_type",
        upper(trim(col("transaction_type")))
    )

    # Preserve original source value for auditing.
    .withColumn(
        "quantity_raw",
        col("quantity").cast("string")
    )

    # Safely convert malformed values to NULL.
    .withColumn(
        "quantity",
        expr(
            "try_cast(quantity AS INT)"
        )
    )

    .withColumn(
        "transaction_timestamp",
        expr(
            "try_to_timestamp(transaction_timestamp)"
        )
    )
)

In [0]:

joined_df = (
    inventory_prepared_df.alias("inventory")

    .join(
        product_lookup_df.alias("product"),
        on="product_id",
        how="left"
    )
)

## Business-quality routing and inventory semantics

Rules validate required identifiers, timestamps, transaction types, quantities, and product matches. Rejected rows retain their original quantity and reason; valid rows translate receipt, return, sale, and adjustment events into signed inventory and value changes for consistent downstream aggregation.

In [0]:
#transaction_id          required
#transaction_timestamp   valid
#product_id              required
#warehouse               required
#transaction_type        valid
#quantity                valid numeric value

#RECEIPT / SALE / RETURN
#quantity > 0

#ADJUSTMENT
#quantity may be positive, negative or zero

#product_id
#must exist in product catalog


from pyspark.sql.functions import when


quality_df = (
    joined_df

    .withColumn(
        "rejection_reason",

        when(
            col("transaction_id").isNull(),
            "Missing transaction_id"
        )

        .when(
            col("transaction_timestamp").isNull(),
            "Invalid or missing transaction timestamp"
        )

        .when(
            col("product_id").isNull(),
            "Missing product_id"
        )

        .when(
            col("warehouse").isNull(),
            "Missing warehouse"
        )

        .when(
            ~col("transaction_type").isin(
                "RECEIPT",
                "SALE",
                "RETURN",
                "ADJUSTMENT"
            ),
            "Invalid transaction type"
        )

        .when(
            col("quantity").isNull(),
            "Invalid or missing quantity"
        )

        .when(
            (
                col("transaction_type").isin(
                    "RECEIPT",
                    "SALE",
                    "RETURN"
                )
            )
            & (col("quantity") <= 0),
            "Quantity must be greater than zero"
        )

        .when(
            col("product_name").isNull(),
            "Product not found in product catalog"
        )

        .otherwise(None)
    )
)

In [0]:

rejected_df = (
    quality_df
        .filter(
            col("rejection_reason").isNotNull()
        )
)


valid_df = (
    quality_df

    .filter(
        col("rejection_reason").isNull()
    )

    .drop("rejection_reason")

    .withColumn(
        "inventory_change",

        when(
            col("transaction_type") == "RECEIPT",
            col("quantity")
        )

        .when(
            col("transaction_type") == "RETURN",
            col("quantity")
        )

        .when(
            col("transaction_type") == "SALE",
            -col("quantity")
        )

        .when(
            col("transaction_type") == "ADJUSTMENT",
            col("quantity")
        )
    )

    .withColumn(
        "inventory_value_change",
        (
            col("inventory_change")
            * col("unit_cost")
        ).cast("decimal(14,2)")
    )

    .withColumn(
        "silver_processing_timestamp",
        lit(ingestion_timestamp).cast("timestamp")
    )
)


## Idempotent Silver persistence

Initial loads create the external Delta targets. Subsequent runs MERGE by `transaction_id`, update only when a newer ingestion timestamp arrives, insert new transactions, and allow additive schema evolution for both valid and rejected datasets.

In [0]:


from delta.tables import DeltaTable


def merge_silver_snapshot(
    source_df,
    target_table,
    target_path,
    merge_key
):
    """
    Creates an external Silver Delta table on the initial load.

    Subsequent executions synchronize the Silver table with
    the complete transformed Bronze snapshot using Delta MERGE.
    """

    if not spark.catalog.tableExists(target_table):

        print(
            f"Initial load. Creating Silver table: "
            f"{target_table}"
        )

        (
            source_df.write
                .format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .option("path", target_path)
                .saveAsTable(target_table)
        )

    else:

        print(
            f"Synchronizing Silver table: "
            f"{target_table}"
        )

        target = DeltaTable.forName(
            spark,
            target_table
        )

        (
            target.alias("target")

                .merge(
                    source_df.alias("source"),
                    f"target.{merge_key} = source.{merge_key}"
                )

                .withSchemaEvolution()

                .whenMatchedUpdateAll(
                    condition=(
                        "source.ingestion_timestamp "
                        "> target.ingestion_timestamp"
                    )
                )

                .whenNotMatchedInsertAll()

                .whenNotMatchedBySourceDelete()

                .execute()
        )

    print(
        f"Silver synchronization completed: "
        f"{target_table}"
    )

In [0]:

merge_silver_snapshot(
    source_df=valid_df,
    target_table=silver_table,
    target_path=silver_path,
    merge_key="transaction_id"
)

merge_silver_snapshot(
    source_df=rejected_df,
    target_table=rejected_table,
    target_path=rejected_path,
    merge_key="transaction_id"
)